# exp-20260913-04 — a screening metric with a usable noise floor

**The problem.** Our only instrument is 58 gold studies with σ = 0.0555, so the significance bar is
**Δ > 0.1110**. The public leaderboard already showed us a real effect our own bar calls unprovable:
pretrained vs random-init is **+0.046 on the LB** and **+0.030 on gold CV**. We cannot run a research
programme on an instrument that cannot see the effects we are producing.

**The idea.** Score against the ~3,400 weakly-labelled studies instead — 60× the sample size, with
noisier labels. Because the model trains on those labels, the predictions must be **out-of-fold**.

**Known-answer test.** Run both arms (pretrained, random-init) through the same folds. The LB says
pretrained wins by 0.046. A useful instrument must (a) agree, and (b) have a bar tight enough to
call it.

**What this metric is not:** agreement with weak labels, which exp-03 showed are a mention detector
with a pooled false-positive rate of 0.66. Its absolute value is not a gold AUC and must never be
reported as "our score".


In [ ]:
# ================================================================== CONFIG — the only cell to edit
from __future__ import annotations
import glob, json, os, re, time, unicodedata, warnings
from pathlib import Path
import numpy as np
import pandas as pd

RUN_MODE = "full"          # "smoke" (minutes, proves it runs) | "full" (the baseline) | "submit" (inference only)

# ---- competition constants (rules.md) -----------------------------------------------------------
TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
N_LABEL = len(TARGETS)
SUBMISSION_NAME = "submission.csv"        # rules.md: hard requirement
KAGGLE_LIMIT_H  = 9.0                     # rules.md: CPU or GPU notebook <= 9 h
WORKING_LIMIT_H = 6.75                    # rules.md: 9 h minus 25% headroom

# ---- versioned artefacts (rules.md: cache is keyed by preprocessing version) --------------------
LABELLER_VERSION  = "v1-keyword"          # weak-label rules; bump when the labeller changes
PREPROC_VERSION   = "p1"                  # bump on ANY change to the DICOM -> tensor path
EXPERIMENT_ID     = "baseline-v1"

# ---- study -> tensor geometry -------------------------------------------------------------------
SLOTS       = [("Sagittal", 1), ("Coronal", 1), ("Axial", 1)]   # (plane, prefer fluid-sensitive)
N_SLOT      = len(SLOTS)
N_TRIPLET   = 4            # windows per slot; a window = 3 adjacent slices stacked as channels
IMG         = 192
CROP_MM     = 130.0
SLICE_BAND  = (0.15, 0.85)
K           = N_SLOT * N_TRIPLET

# ---- model / training ---------------------------------------------------------------------------
BACKBONE     = "resnet18"
PRETRAINED   = True        # with internet off this needs an attached weights dataset; see below
EPOCHS       = 4           # FIXED. Never chosen by looking at gold (rules.md hard rule 2)
BATCH        = 8
LR_HEAD      = 3e-4
LR_BACKBONE  = 1e-4
NUM_WORKERS  = 2

# ---- evaluation protocol (rules.md: statistical rules) ------------------------------------------
N_SEEDS   = {"smoke": 1, "full": 3, "submit": 1}[RUN_MODE]   # training seeds -> seed variance
SEEDS     = [2026, 2027, 2028][:N_SEEDS]
N_FOLDS   = 5              # evaluation folds over the 58 gold studies (NOT training folds)
EVAL_REPEATS = 5           # fold reshuffles; sigma comes from (seed x repeat x fold) cells
# Measured on a T4: with a warm cache an epoch costs ~2 s per 120 studies, so training is
# decode-bound, not compute-bound. With the prebuilt cache attached, use every weakly-labelled
# study — the old 1200 cap only ever existed to fit a decode budget the cache removes.
MAX_TRAIN_STUDIES = {"smoke": 120, "full": 10_000, "submit": 0}[RUN_MODE]
# Prebuilt tensor cache: the output of notebooks/cache-build-p1.ipynb, attached as a data source.
# Kaggle mounts a notebook's output at /kaggle/input/<notebook-slug>/ (plus our cache_<ver> subdir).
CACHE_INPUT_DIRS  = ["/kaggle/input/rsna-knee-cache-build-p1/cache_p1",
                     "/kaggle/input/rsna-knee-cache-build-p1",
                     "/kaggle/input/rsna-knee-cache-p1"]

# ---- offline weights (rules.md: internet is disabled in the rerun) ------------------------------
# Backbone weights ship as an attached dataset because the rerun cannot download anything.
# Dataset: kaggle.com/datasets/vaibhav486/timm-backbones-offline (Apache-2.0, redistributable —
# required by the winners' obligation to publish weights).
TIMM_OFFLINE_DIRS = ["/kaggle/input/timm-backbones-offline", "/kaggle/input/timm-weights"]
BACKBONE_WEIGHTS = {                      # timm model name -> file in the dataset above
    "resnet18":           "resnet18.a1_in1k.bin",
    "resnet34":           "resnet34.a1_in1k.bin",
    "tf_efficientnet_b0": "tf_efficientnet_b0.ns_jft_in1k.bin",
    "convnext_tiny":      "convnext_tiny.in12k_ft_in1k.bin",
}
CHECKPOINT_DIRS   = ["/kaggle/input/rsna-knee-baseline-v1"]   # our own trained weights, for "submit"

# ---- paths ---------------------------------------------------------------------------------------
def find_root() -> Path:
    for c in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
              Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"), Path(".")]:
        if (c / "train.csv").exists() or list(c.glob("train*.csv")):
            return c
    raise FileNotFoundError("Competition data not found; set ROOT by hand.")

def find_csv(root: Path, stem: str) -> Path:
    exact = root / f"{stem}.csv"
    if exact.exists():
        return exact
    hits = sorted(c for c in root.glob(f"{stem}*.csv") if "_series" not in c.name)
    if not hits:
        raise FileNotFoundError(f"{stem}.csv not found under {root}")
    return hits[0]

ROOT  = find_root()
WORK  = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./work")
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / f"cache_{PREPROC_VERSION}"          # version in the path: a stale cache cannot be reused
CACHE.mkdir(parents=True, exist_ok=True)


def discover_input(pattern: str, want_dir: bool = False) -> list[Path]:
    """Find something under /kaggle/input without guessing the mount layout.

    Kaggle has mounted attachments at BOTH /kaggle/input/<slug> and the nested
    /kaggle/input/{datasets,notebooks,competitions}/<owner>/<slug>/[version]/ — and which one you
    get is not under our control. Hardcoding either cost a wasted GPU hour once already, so search
    instead and print what was found."""
    root = Path("/kaggle/input")
    if not root.exists():
        return []
    hits = [p for p in sorted(root.glob(pattern)) if (p.is_dir() if want_dir else p.is_file())]
    return hits


# A prebuilt cache (attached notebook output) is searched first, then our own writable one.
_cache_hits = [Path(d) for d in CACHE_INPUT_DIRS if Path(d).exists()]
_cache_hits += [d for d in discover_input(f"**/cache_{PREPROC_VERSION}", want_dir=True)
                if d not in _cache_hits and d != CACHE]
CACHE_READ = _cache_hits + [CACHE]

T_START = time.time()
def elapsed_h() -> float:
    return (time.time() - T_START) / 3600.0

def budget_check(stage: str) -> None:
    """rules.md hard rule 7: stay inside the 6.75 h working limit, loudly."""
    h = elapsed_h()
    print(f"[budget] {stage}: {h:.2f} h of {WORKING_LIMIT_H} h used ({h / KAGGLE_LIMIT_H:.0%} of the Kaggle cap)")
    if h > WORKING_LIMIT_H:
        warnings.warn(f"OVER THE WORKING BUDGET at '{stage}' — this configuration is not submittable.")

np.random.seed(SEEDS[0])
print(f"run mode   : {RUN_MODE}   seeds={SEEDS}   epochs={EPOCHS}")
print(f"data root  : {ROOT.resolve()}")
print(f"work dir   : {WORK.resolve()}")
print(f"cache      : {CACHE.name}  (preproc {PREPROC_VERSION}, labeller {LABELLER_VERSION})")
print(f"per study  : {K} windows ({N_SLOT} slots x {N_TRIPLET} triplets) at {IMG}x{IMG}")


In [ ]:
test = pd.read_csv(find_csv(ROOT, "test"))
test["StudyInstanceUID"] = test["StudyInstanceUID"].astype(str)

def write_submission(frame: pd.DataFrame) -> Path:
    """Single place that writes the file, so the contract is enforced in one spot."""
    out = frame.copy()
    out["StudyInstanceUID"] = out["StudyInstanceUID"].astype(str)
    out = out[["StudyInstanceUID"] + TARGETS]                 # exact column order
    out[TARGETS] = out[TARGETS].astype(float).clip(0.0, 1.0)  # exact range
    out[TARGETS] = out[TARGETS].fillna(0.5)                   # never a NaN
    path = WORK / SUBMISSION_NAME
    out.to_csv(path, index=False)
    return path

fallback = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"]})
for c in TARGETS:
    fallback[c] = 0.5
print("fallback submission written to:", write_submission(fallback))
print(f"test rows visible here: {len(test)}  (the hidden test set is substituted at rerun)")


In [ ]:
train        = pd.read_csv(find_csv(ROOT, "train"))
train_series = pd.read_csv(find_csv(ROOT, "train_series"))
test_series  = pd.read_csv(find_csv(ROOT, "test_series"))
for df in (train, test, train_series, test_series):
    for col in ("StudyInstanceUID", "SeriesInstanceUID"):
        if col in df.columns:
            df[col] = df[col].astype(str)

gold_mask = train[TARGETS].notna().all(axis=1)
gold = train.loc[gold_mask].reset_index(drop=True)
Y_GOLD = gold[TARGETS].values.astype(int)          # [58, 12] — the only ground truth we own

pos = pd.Series(Y_GOLD.sum(0), index=TARGETS)
print(f"gold studies: {len(gold)}   report-only: {(~gold_mask).sum()}")
print("\npositives per target among the gold studies:")
print(pos.to_string())
print(f"\nrarest target: {pos.idxmin()} with {pos.min()} positives -> "
      f"{pos.min() / N_FOLDS:.1f} expected positives per fold. "
      "This is why undefined (fold, label) cells are unavoidable and must be counted.")


In [ ]:
def normalise(text: str) -> str:
    t = unicodedata.normalize("NFKD", str(text))
    t = "".join(c for c in t if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", t.lower())

NEG = (r"(?:no |not |without |absence of |negative for |intact |normal |unremarkable |ohne |"
       r"kein[e]?[nrms]? |unauff|sin |ausencia|geen |zonder |normale |normaal |bez |uredn|"
       r"nema |sans |pas de |absence)")

PATTERNS = {
    "ACL": r"(acl|anterior cruciate|lca|vkb|ligamento cruzado anterior|voorste kruisband|"
           r"kruisband anterior|prednj[ei] krizn|kreuzband(?:ruptur)?\s*(?:vorder)?|vorderes kreuzband)",
    "MCL": r"(mcl|medial collateral|ligamento colateral medial|innenband|mediale[nr]? kollateralband|"
           r"mediale collaterale|medijalni kolateralni)",
    "Medial Meniscus": r"(medial meniscus|menisco (?:interno|medial)|innenmeniskus|mediale meniscus|"
                       r"medijalni menisk|meniscus medialis|meniscus internus)",
    "Lateral Meniscus": r"(lateral meniscus|menisco (?:externo|lateral)|aussenmeniskus|laterale meniscus|"
                        r"lateralni menisk|meniscus lateralis)",
    "Medial OA": None, "Lateral OA": None, "PF OA": None,       # compartment co-occurrence, below
    "Effusion": r"(effusion|derrame|gelenkerguss|ergus[s]?|hydrops|izljev|epanchement|joint fluid|"
                r"gewrichtsvocht|vocht)",
    "Synovitis": r"(synovit\w*|sinovit\w*|synovialit\w*|sinovij\w*|"
                 r"synovial\w* (?:proliferation|thickening|verdikking|hypertroph\w*)|"
                 r"proliferacij\w* sinovij|pannus|synoviale? reizung)",
    "Baker's": r"(baker|popliteal cyst|quiste de baker|bakerzyste|baker-zyste|bakerova cist|kyste de baker)",
    "Contusion": r"(bone (?:marrow )?(?:contusion|bruise|oedema|edema)|contusion|knochenmarkod|kontuzij|"
                 r"botcontusie|edema oseo)",
    "Fracture": r"(fracture|fractur|fraktur|fisura osea|prijelom|breuk|fractuur|avulsion)",
}
ABNORMAL = (r"(tear|rupt|riss|scheur|rotura|lesion|desgarr|lasion|laesion|degenerativ|signal|"
            r"tearing|ruptura|insuffizienz|discontinu|abnormal)")
NEEDS_ABNORMAL = {"ACL", "MCL", "Medial Meniscus", "Lateral Meniscus"}
OA_WORD = (r"(osteoarthrit\w*|arthros\w*|artros\w*|artroz\w*|gonarthros\w*|osteoartr\w*|chondral loss|"
           r"cartilage loss|knorpel\w*|chondropath\w*|kraakbeen\w*|hrskavic\w*|osteophyt\w*|osteofit\w*|"
           r"degenerative (?:change|veranderung)\w*|denudacij\w*)")
COMPARTMENT = {
    "Medial OA":  r"(medial\w*|mediaal|medijaln\w*|intern[oa]|innen\w*|inner)",
    "Lateral OA": r"(lateral\w*|lateraal|lateraln\w*|extern[oa]|aussen\w*|outer)",
    "PF OA":      r"(patell\w*|patel\w*|femoropatel\w*|retropatell\w*|trochlea\w*|trohlej\w*)",
}
WINDOW = 60

def _oa_label(t: str, key: str) -> float:
    pos = neg = 0
    for m in re.finditer(OA_WORD, t):
        s, e = m.span()
        ctx = t[max(0, s - WINDOW):e + WINDOW]
        if not re.search(COMPARTMENT[key], ctx):
            continue
        if key != "PF OA" and re.search(r"(?:patell|patel|trochlea|trohlej)", ctx):
            continue
        if re.search(NEG + r"[^.]{0,25}$", t[max(0, s - 45):s]):
            neg += 1
        else:
            pos += 1
    return 1.0 if pos else (0.0 if neg else np.nan)

def label_report(text: str) -> dict:
    """{finding: 1.0 | 0.0 | nan}. nan = the report does not say — NOT a zero (rules.md)."""
    t = normalise(text)
    out = {k: _oa_label(t, k) for k in COMPARTMENT}
    for key, pattern in PATTERNS.items():
        if pattern is None:
            continue
        pos = neg = 0
        for m in re.finditer(pattern, t):
            s, e = m.span()
            left, right = t[max(0, s - 45):s], t[e:e + 80]
            negated = (re.search(NEG + r"[^.]{0,25}$", left)
                       or re.search(r"^\W{0,4}(?:" + NEG + r"|ist intakt|intacto|intact)", right))
            if key in NEEDS_ABNORMAL:
                near_abnormal = re.search(ABNORMAL, right[:60]) or re.search(ABNORMAL + r"[^.]{0,30}$", left)
                if not near_abnormal:
                    if re.search(r"^\W{0,6}(?:" + NEG + r"|intact|normal)", right):
                        neg += 1
                    continue
            if negated:
                neg += 1
            else:
                pos += 1
        out[key] = 1.0 if pos else (0.0 if neg else np.nan)
    return out

weak_labels = pd.DataFrame([label_report(r) for r in train["Report"]])[TARGETS]
weak_labels.insert(0, "StudyInstanceUID", train["StudyInstanceUID"].values)
print("label coverage (share of studies the report can decide):")
print((weak_labels[TARGETS].notna().mean() * 100).round(1).to_string())


In [ ]:
import pydicom
import cv2
cv2.setNumThreads(1)                       # parallelism is across studies, not inside OpenCV

SERIES_DIR_TRAIN = ROOT / "train_series"
SERIES_DIR_TEST  = ROOT / "test_series"
HAVE_IMAGES = SERIES_DIR_TRAIN.exists() or SERIES_DIR_TEST.exists()
print("image directories present:", HAVE_IMAGES)
if not HAVE_IMAGES:
    print("  -> metadata-only environment: training is skipped, the fallback submission stands.")

series_by_study      = {k: v.to_dict("records") for k, v in train_series.groupby("StudyInstanceUID")}
series_by_study_test = {k: v.to_dict("records") for k, v in test_series.groupby("StudyInstanceUID")}

def pick_series(rows, plane, fluid, used):
    cand = [r for r in rows if r["Anatomical_Plane"] == plane and r["SeriesInstanceUID"] not in used]
    pref = [r for r in cand if int(r.get("Fluid_Sensitive", 0) or 0) == fluid]
    return (pref or cand or [None])[0]

def ordered_slices(series_dir: Path):
    keyed = []
    for f in glob.glob(str(series_dir / "*.dcm")):
        try:
            hdr = pydicom.dcmread(f, stop_before_pixels=True)
            pos = int(getattr(hdr, "InstanceNumber", 0) or 0)
            spacing = float(hdr.PixelSpacing[0]) if hasattr(hdr, "PixelSpacing") else 0.0
        except Exception:
            continue
        keyed.append((pos, f, spacing))
    keyed.sort()
    return [(f, s) for _, f, s in keyed]

def read_pixels(path):
    try:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array.astype(np.float32)
        if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
            arr = arr.max() - arr
        return arr
    except Exception:
        return None

def crop_and_resize(arr, spacing):
    h, w = arr.shape
    if spacing <= 0:
        spacing = CROP_MM / max(h, w)
    side = min(int(round(CROP_MM / spacing)), h, w)
    y0, x0 = (h - side) // 2, (w - side) // 2
    return cv2.resize(arr[y0:y0 + side, x0:x0 + side], (IMG, IMG), interpolation=cv2.INTER_AREA)

def build_study(study_uid: str, rows, series_dir: Path):
    """-> (uint8 [K,3,IMG,IMG], bool [K]). A missing or unreadable slot is zeros with mask False."""
    volume = np.zeros((K, 3, IMG, IMG), np.uint8)
    mask = np.zeros(K, bool)
    used, w = set(), 0
    for plane, fluid in SLOTS:
        record = pick_series(rows, plane, fluid, used)
        if record is None:
            w += N_TRIPLET; continue
        used.add(record["SeriesInstanceUID"])
        files = ordered_slices(series_dir / study_uid / record["SeriesInstanceUID"])
        n = len(files)
        if n == 0:
            w += N_TRIPLET; continue
        lo, hi = int(n * SLICE_BAND[0]), max(int(n * SLICE_BAND[1]) - 1, 0)
        centres = np.linspace(lo, max(hi, lo), N_TRIPLET).round().astype(int)
        med = float(np.median([s for _, s in files if s > 0]) if any(s > 0 for _, s in files) else 0.0)
        for centre in centres:
            idx = [int(np.clip(centre + d, 0, n - 1)) for d in (-1, 0, 1)]
            planes, spacings = [], []
            for i in idx:
                path, spacing = files[i]
                planes.append(read_pixels(path))
                spacings.append(spacing if spacing > 0 else med)
            present = [p for p in planes if p is not None]
            if not present:
                w += 1; continue
            low, high = np.percentile(np.concatenate([p.ravel() for p in present]), [2.0, 98.0])
            for c, (p, spacing) in enumerate(zip(planes, spacings)):
                if p is None:
                    continue
                norm = np.clip((p - low) / (high - low + 1e-6), 0, 1)
                volume[w, c] = (crop_and_resize(norm, spacing) * 255).astype(np.uint8)
            mask[w] = True
            w += 1
    return volume, mask

def cached_study(study_uid: str, rows, series_dir: Path):
    """DICOM decoding, not the GPU, is the bottleneck — build once per (study, PREPROC_VERSION)."""
    for d in CACHE_READ:                          # attached prebuilt cache first, then our own
        path = d / f"{study_uid}.npz"
        if path.exists():
            try:
                with np.load(path) as z:
                    return z["volume"], z["mask"]
            except Exception:
                if d == CACHE:
                    path.unlink(missing_ok=True)  # corrupt entry we own: rebuild rather than crash
    path = CACHE / f"{study_uid}.npz"
    volume, mask = build_study(study_uid, rows, series_dir)
    np.savez_compressed(path, volume=volume, mask=mask)
    return volume, mask


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torch.utils.data import Dataset, DataLoader

def pick_device() -> tuple[str, str]:
    """Kaggle hands out P100s (sm_60) that the installed PyTorch no longer supports — the failure
    is a `no kernel image is available` CUDA error thrown deep inside the first conv, minutes into
    a run. Detect the mismatch up front and degrade to CPU instead of crashing."""
    if not torch.cuda.is_available():
        return "cpu", "no CUDA device"
    name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    cap = major * 10 + minor
    supported = sorted(int(a.split("_")[1]) for a in torch.cuda.get_arch_list() if a.startswith("sm_"))
    if supported and cap < min(supported):
        warnings.warn(
            f"{name} is sm_{cap}; this PyTorch supports sm_{supported}. FALLING BACK TO CPU. "
            "Switch the notebook accelerator to 'GPU T4 x2' in the Kaggle UI (Settings -> "
            "Accelerator) — the CLI cannot select the GPU type — then re-run.")
        return "cpu", f"{name} (sm_{cap}) unsupported by this torch build"
    return "cuda", name

DEVICE, DEVICE_NOTE = pick_device()
USE_AMP = DEVICE == "cuda"
try:
    from torch.amp import GradScaler as _GS, autocast as _AC
    def make_scaler(): return _GS("cuda", enabled=USE_AMP)
    def amp_autocast(): return _AC("cuda", enabled=USE_AMP)
except ImportError:
    def make_scaler(): return torch.cuda.amp.GradScaler(enabled=USE_AMP)
    def amp_autocast(): return torch.cuda.amp.autocast(enabled=USE_AMP)

print("device:", DEVICE, "|", DEVICE_NOTE)
if DEVICE == "cpu" and torch.cuda.is_available():
    print("  !! a GPU is attached but unusable — this run will be slow and is NOT the baseline")

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

def set_all_seeds(seed: int) -> None:
    """rules.md hard rule 5: same notebook, same input, same number."""
    np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE == "cuda":
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def find_offline_weights() -> Path | None:
    """The attached weights file for BACKBONE, wherever Kaggle decided to mount it."""
    fname = BACKBONE_WEIGHTS.get(BACKBONE)
    if not fname:
        return None
    for d in TIMM_OFFLINE_DIRS:                     # the paths we expect
        f = Path(d) / fname
        if f.exists():
            return f
    hits = discover_input(f"**/{fname}")            # ...and anywhere else it actually landed
    return hits[0] if hits else None


def make_backbone():
    """Offline first: the rerun has no internet, so weights come from an attached dataset.
    The load is *verified* — a silently-partial load would look pretrained and train like noise."""
    net = timm.create_model(BACKBONE, pretrained=False, num_classes=0, in_chans=3, global_pool="avg")
    if not PRETRAINED:
        return net, False

    f = find_offline_weights()
    if f is not None:
        sd = torch.load(f, map_location="cpu", weights_only=True)
        missing, unexpected = net.load_state_dict(sd, strict=False)
        # num_classes=0 drops the classifier, so those keys are expected to be unexpected.
        stray = [k for k in unexpected if not re.match(r"^(fc|classifier|head)\.", k)]
        if missing or stray:
            warnings.warn(f"offline weights loaded PARTIALLY from {f.name}: "
                          f"{len(missing)} missing, stray unexpected {stray[:5]} — treat as random init.")
            return net, False
        print(f"backbone weights: {f} (offline, {len(sd)} tensors, 0 missing)")
        return net, True

    try:                                   # training notebooks may have internet; submissions never do
        net = timm.create_model(BACKBONE, pretrained=True, num_classes=0, in_chans=3, global_pool="avg")
        print("backbone weights: downloaded from the hub (internet is ON — not the submission path)")
        return net, True
    except Exception as e:
        warnings.warn(f"NO pretrained weights ({type(e).__name__}) — RANDOM INIT. Attach "
                      f"{TIMM_OFFLINE_DIRS[0]}; a random-init score is not a baseline (plans/baseline-v1.md).")
        return net, False

class KneeStudyDataset(Dataset):
    """One item = one study: K windows, a window-validity mask, 12 possibly-unknown labels."""
    def __init__(self, uids, labels, series_map, series_dir, train_mode=False):
        self.uids, self.labels = list(uids), np.asarray(labels, np.float32)
        self.series_map, self.series_dir, self.train_mode = series_map, series_dir, train_mode
    def __len__(self):
        return len(self.uids)
    def __getitem__(self, i):
        uid = self.uids[i]
        volume, mask = cached_study(uid, self.series_map.get(uid, []), self.series_dir)
        x = torch.from_numpy(volume.astype(np.float32) / 255.0)
        if self.train_mode and np.random.rand() < 0.5:
            x = torch.flip(x, dims=[-1])
        x = (x - IMAGENET_MEAN) / IMAGENET_STD
        y = torch.from_numpy(self.labels[i])
        return x, torch.from_numpy(mask.astype(np.float32)), torch.nan_to_num(y), ~torch.isnan(y)

class KneeNet(nn.Module):
    def __init__(self, n_label=N_LABEL, drop=0.2):
        super().__init__()
        self.backbone, self.is_pretrained = make_backbone()
        dim = self.backbone.num_features
        self.norm = nn.LayerNorm(dim)
        self.attn = nn.Sequential(nn.Linear(dim, 256), nn.Tanh(), nn.Dropout(drop),
                                  nn.Linear(256, n_label))
        self.cls_w = nn.Parameter(torch.zeros(n_label, dim))
        self.cls_b = nn.Parameter(torch.zeros(n_label))
        nn.init.trunc_normal_(self.cls_w, std=0.02)
    def forward(self, x, window_mask):
        b, k = x.shape[:2]
        h = self.norm(self.backbone(x.flatten(0, 1)).view(b, k, -1))
        a = self.attn(h).masked_fill(window_mask[:, :, None] < 0.5, float("-inf"))
        empty = window_mask.sum(1) == 0            # a study with no readable window: uniform, not NaN
        if empty.any():
            a[empty] = 0.0
        pooled = torch.einsum("bkn,bkf->bnf", torch.softmax(a, dim=1), h)
        return (pooled * self.cls_w).sum(-1) + self.cls_b

def masked_bce(logits, targets, target_mask):
    """Unknown label cells contribute nothing — they are masked, never filled (rules.md)."""
    loss = F.binary_cross_entropy_with_logits(logits, targets, reduction="none") * target_mask.float()
    return loss.sum() / target_mask.float().sum().clamp(min=1.0)


In [ ]:
usable = weak_labels[TARGETS].notna().any(axis=1) & ~gold_mask.values
all_train_uids = weak_labels.loc[usable, "StudyInstanceUID"].tolist()
all_train_y    = weak_labels.loc[usable, TARGETS].values.astype(np.float32)
gold_uids      = gold["StudyInstanceUID"].tolist()

print(f"weakly-labelled studies available : {len(all_train_uids)}")
print(f"known label cells among them      : {np.isfinite(all_train_y).mean() * 100:.1f}%")
print(f"using this run                    : {min(MAX_TRAIN_STUDIES, len(all_train_uids))}")
print(f"measuring on                      : {len(gold_uids)} gold studies (never trained on)")


def train_one_seed(seed: int):
    """Train once; return gold probabilities [58, 12] and timing. Deterministic given `seed`."""
    set_all_seeds(seed)
    rng = np.random.default_rng(seed)
    uids, ys = all_train_uids, all_train_y
    if len(uids) > MAX_TRAIN_STUDIES:
        keep = rng.choice(len(uids), MAX_TRAIN_STUDIES, replace=False)
        uids = [uids[i] for i in keep]
        ys = ys[keep]

    train_dl = DataLoader(KneeStudyDataset(uids, ys, series_by_study, SERIES_DIR_TRAIN, True),
                          batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
    gold_dl  = DataLoader(KneeStudyDataset(gold_uids, np.full((len(gold_uids), N_LABEL), np.nan, np.float32),
                                           series_by_study, SERIES_DIR_TRAIN),
                          batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)

    model = KneeNet().to(DEVICE)
    head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
    opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                             {"params": head, "lr": LR_HEAD}], weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD],
                                                total_steps=max(EPOCHS * len(train_dl), 1), pct_start=0.2)
    scaler = make_scaler()

    t0 = time.time()
    for epoch in range(EPOCHS):
        model.train()
        running = seen = 0
        te = time.time()
        for x, wm, y, ym in train_dl:
            x, wm, y, ym = x.to(DEVICE), wm.to(DEVICE), y.to(DEVICE), ym.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with amp_autocast():
                loss = masked_bce(model(x, wm), y, ym)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            running += loss.item() * x.size(0); seen += x.size(0)
        # Printed for monitoring only — NOT used to pick a checkpoint.
        print(f"  seed {seed} epoch {epoch + 1}/{EPOCHS}  loss {running / max(seen, 1):.4f}  "
              f"({time.time() - te:.0f}s)")

    model.eval()
    probs = []
    with torch.no_grad():
        for x, wm, _, _ in gold_dl:
            with amp_autocast():
                probs.append(torch.sigmoid(model(x.to(DEVICE), wm.to(DEVICE))).float().cpu().numpy())
    probs = np.concatenate(probs)
    torch.save(model.state_dict(), WORK / f"knee_{EXPERIMENT_ID}_seed{seed}.pt")
    np.save(WORK / f"gold_probs_seed{seed}.npy", probs)
    return model, probs, time.time() - t0


In [ ]:
# ---- preflight: fail FAST, before spending an hour on a run that cannot be a baseline ----------
def preflight(strict: bool) -> None:
    """plans/baseline-v1.md kill criteria, enforced in code. A `full` run that trains on random
    init or re-decodes from scratch has already wasted its GPU hour by the time you read the log —
    so check the attachments before the first epoch, not after."""
    problems = []
    if find_offline_weights() is None:
        problems.append(f"no backbone weights in {TIMM_OFFLINE_DIRS} — training would start from "
                        "RANDOM INIT, which is an explicit kill criterion, not a baseline")
    prebuilt = [d for d in CACHE_READ if d != CACHE and any(d.glob("*.npz"))]
    if not prebuilt:
        problems.append(f"no prebuilt tensor cache found in {CACHE_INPUT_DIRS} — every epoch-1 "
                        "would re-decode ~3,400 studies (~1 h)")
    if not problems:
        print("preflight: weights and cache both attached")
        return
    msg = "PREFLIGHT FAILED:\n  - " + "\n  - ".join(problems)
    if strict:
        raise RuntimeError(msg + "\n\nAttach the sources and re-run. On Kaggle, a NEWLY CREATED "
                           "notebook sometimes runs before its data sources mount — push a second "
                           "version and check /kaggle/input before trusting the run.")
    warnings.warn(msg + "\n(continuing because RUN_MODE is not 'full')")

if HAVE_IMAGES:
    print("attached inputs:", sorted(p.name for p in Path("/kaggle/input").glob("*")) or "none")
    print("  weights found :", find_offline_weights())
    print("  cache dirs    :", [str(d) for d in CACHE_READ])
    preflight(strict=(RUN_MODE == "full"))


## Fold setup

Folds are over **studies**, so a study's own tensors can never appear in both training and
evaluation. The 58 gold studies are excluded entirely — they are not part of this instrument.


In [ ]:
N_FOLDS_SCREEN = 3
ARMS = {"pretrained": True, "random_init": False}      # the only difference between arms
SCREEN_SEED = 4242

uids = np.array(all_train_uids)
Y_WEAK = all_train_y.copy()                            # [N, 12] with NaN for undecided
assert not set(uids) & set(gold_uids), "gold leaked into the screening pool"

rng = np.random.default_rng(SCREEN_SEED)
perm = rng.permutation(len(uids))
fold_of = np.zeros(len(uids), int)
for i, idx in enumerate(perm):
    fold_of[idx] = i % N_FOLDS_SCREEN

print(f"screening pool: {len(uids)} studies, {np.isfinite(Y_WEAK).mean()*100:.1f}% of label cells decided")
print("fold sizes:", np.bincount(fold_of).tolist())
print("decided cells per target:")
print(pd.Series(np.isfinite(Y_WEAK).sum(0), index=TARGETS).to_string())


## Train the folds

Each fold trains on the other folds and predicts its own held-out studies, for both arms. Identical
data, folds, seed and schedule — only the backbone initialisation differs.


In [ ]:
def train_fold(train_idx, pred_idx, pretrained: bool, seed: int):
    """Train on train_idx, return probabilities for pred_idx."""
    set_all_seeds(seed)
    global PRETRAINED
    saved, PRETRAINED = PRETRAINED, pretrained          # make_backbone reads the global
    try:
        dl_tr = DataLoader(KneeStudyDataset(uids[train_idx], Y_WEAK[train_idx], series_by_study,
                                            SERIES_DIR_TRAIN, True),
                           batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS, drop_last=True)
        dl_pr = DataLoader(KneeStudyDataset(uids[pred_idx],
                                            np.full((len(pred_idx), N_LABEL), np.nan, np.float32),
                                            series_by_study, SERIES_DIR_TRAIN),
                           batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
        model = KneeNet().to(DEVICE)
        assert model.is_pretrained == pretrained, \
            f"arm asked for pretrained={pretrained} but got {model.is_pretrained} — not a clean comparison"
        head = [p for n, p in model.named_parameters() if not n.startswith("backbone.")]
        opt = torch.optim.AdamW([{"params": model.backbone.parameters(), "lr": LR_BACKBONE},
                                 {"params": head, "lr": LR_HEAD}], weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[LR_BACKBONE, LR_HEAD],
                                                    total_steps=max(EPOCHS * len(dl_tr), 1), pct_start=0.2)
        scaler = make_scaler()
        for epoch in range(EPOCHS):
            model.train()
            t0 = time.time()
            for x, wm, y, ym in dl_tr:
                x, wm, y, ym = x.to(DEVICE), wm.to(DEVICE), y.to(DEVICE), ym.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                with amp_autocast():
                    loss = masked_bce(model(x, wm), y, ym)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            print(f"    epoch {epoch+1}/{EPOCHS} ({time.time()-t0:.0f}s)", flush=True)
        model.eval()
        out = []
        with torch.no_grad():
            for x, wm, _, _ in dl_pr:
                with amp_autocast():
                    out.append(torch.sigmoid(model(x.to(DEVICE), wm.to(DEVICE))).float().cpu().numpy())
        gold_dl = DataLoader(KneeStudyDataset(gold_uids, np.full((len(gold_uids), N_LABEL), np.nan, np.float32),
                                              series_by_study, SERIES_DIR_TRAIN),
                             batch_size=BATCH, shuffle=False, num_workers=NUM_WORKERS)
        gold_out = []
        with torch.no_grad():
            for x, wm, _, _ in gold_dl:
                with amp_autocast():
                    gold_out.append(torch.sigmoid(model(x.to(DEVICE), wm.to(DEVICE))).float().cpu().numpy())
        return np.concatenate(out), np.concatenate(gold_out)
    finally:
        PRETRAINED = saved


In [ ]:
if HAVE_IMAGES:
    preflight(strict=True)

oof = {a: np.full((len(uids), N_LABEL), np.nan, np.float32) for a in ARMS}
gold_pred = {a: [] for a in ARMS}
t_start = time.time()

if HAVE_IMAGES:
    for arm, pretrained in ARMS.items():
        for f in range(N_FOLDS_SCREEN):
            pred_idx = np.where(fold_of == f)[0]
            train_idx = np.where(fold_of != f)[0]
            print(f"\n[{arm}] fold {f}: train {len(train_idx)}, predict {len(pred_idx)}", flush=True)
            p, gp = train_fold(train_idx, pred_idx, pretrained, seed=SCREEN_SEED + f)
            oof[arm][pred_idx] = p
            gold_pred[arm].append(gp)
            budget_check(f"{arm} fold {f}")
    for a in ARMS:
        np.save(WORK / f"screen_oof_{a}.npy", oof[a])
        np.save(WORK / f"screen_goldpred_{a}.npy", np.mean(gold_pred[a], axis=0))
    print(f"\ntotal training wall clock: {(time.time()-t_start)/3600:.2f} h")


## Score, and measure the instrument's own noise

`screen_auc` is the macro AUC against the weak labels over held-out predictions. The noise floor
comes from bootstrapping **studies** (not cells), which is the unit that actually varies.


In [ ]:
from sklearn.metrics import roc_auc_score   # the labeller-AUC cell that normally
# imports this is not part of this notebook — exp-04 v1 trained 6 folds and then died here
def full_macro_auc(y: np.ndarray, p: np.ndarray) -> float:
    """Macro AUC on all 58 at once — the estimate most comparable to the leaderboard."""
    aucs = [roc_auc_score(y[:, j], p[:, j]) for j in range(y.shape[1]) if len(np.unique(y[:, j])) > 1]
    return float(np.mean(aucs))


def screen_auc(y_weak, p, idx=None):
    """Macro AUC vs weak labels over the given studies; undecided cells are skipped, not imputed."""
    sel = np.arange(len(y_weak)) if idx is None else idx
    aucs = []
    for j in range(N_LABEL):
        yy, pp = y_weak[sel, j], p[sel, j]
        m = np.isfinite(yy) & np.isfinite(pp)
        if m.sum() < 20 or len(np.unique(yy[m])) < 2:
            continue
        aucs.append(roc_auc_score(yy[m], pp[m]))
    return float(np.mean(aucs)) if aucs else float("nan")


def bootstrap(y_weak, pa, pb, n=200, seed=0):
    """Resample studies; return the per-arm scores and the PAIRED delta distribution."""
    rng = np.random.default_rng(seed)
    n_study = len(y_weak)
    A, B, D = [], [], []
    for _ in range(n):
        idx = rng.integers(0, n_study, n_study)
        a, b = screen_auc(y_weak, pa, idx), screen_auc(y_weak, pb, idx)
        A.append(a); B.append(b); D.append(a - b)
    return np.array(A), np.array(B), np.array(D)


if HAVE_IMAGES:
    res = {}
    for a in ARMS:
        res[a] = screen_auc(Y_WEAK, oof[a])
        g = np.load(WORK / f"screen_goldpred_{a}.npy")
        res[a + "_gold"] = full_macro_auc(Y_GOLD, g)
    A, B, D = bootstrap(Y_WEAK, oof["pretrained"], oof["random_init"])

    print(f"screening metric (OOF vs weak labels, {len(uids)} studies)")
    print(f"  pretrained  : {res['pretrained']:.4f}")
    print(f"  random init : {res['random_init']:.4f}")
    print(f"  delta       : {res['pretrained'] - res['random_init']:+.4f}")
    print(f"  paired bootstrap sigma(delta) : {D.std():.4f}   -> 2 sigma bar = {2*D.std():.4f}")
    print(f"  delta 95% CI                  : [{np.percentile(D,2.5):+.4f}, {np.percentile(D,97.5):+.4f}]")
    print(f"  per-arm bootstrap sigma       : {A.std():.4f} / {B.std():.4f}")
    print(f"\nsame two arms on the 58 gold studies (this run's fold-mean models)")
    print(f"  pretrained  : {res['pretrained_gold']:.4f}")
    print(f"  random init : {res['random_init_gold']:.4f}")
    print(f"\nreference bars")
    print(f"  gold 2 sigma (baseline run)   : 0.1110")
    print(f"  screening 2 sigma (this run)  : {2*D.std():.4f}")
    print(f"  known answer from the public LB: pretrained - random init = +0.046")


In [ ]:
if HAVE_IMAGES:
    verdict = ("SUCCESS (+1)" if (D.mean() > 0 and 2*D.std() < 0.03) else
               "PARTIAL" if (D.mean() > 0 and 2*D.std() < 0.1110) else "FAILURE (-1)")
    summary = {"experiment_id": "exp-20260913-04", "date": time.strftime("%Y-%m-%d"),
               "n_screen_studies": int(len(uids)), "n_folds": N_FOLDS_SCREEN,
               "screen_auc_pretrained": round(res["pretrained"], 4),
               "screen_auc_random_init": round(res["random_init"], 4),
               "delta": round(float(D.mean()), 4),
               "delta_sigma": round(float(D.std()), 4),
               "screen_2sigma_bar": round(float(2*D.std()), 4),
               "delta_ci95": [round(float(np.percentile(D, 2.5)), 4), round(float(np.percentile(D, 97.5)), 4)],
               "gold_2sigma_bar": 0.1110, "lb_known_answer": 0.046,
               "gold_auc_pretrained": round(res["pretrained_gold"], 4),
               "gold_auc_random_init": round(res["random_init_gold"], 4),
               "hours": round(elapsed_h(), 2), "verdict": verdict}
    with open(WORK / "exp04_results.json", "w") as f:
        json.dump(summary, f, indent=2)
    print(json.dumps(summary, indent=2))
    budget_check("end")
